# Existing-product comparison
-------

This study's map and the four existing products (Sabatini, Munteanu, Kathmann, Schickhofer) are evaluated against the reference labels at the parcel level, using the products' own published predictions with no tuning.

This study is tuned on the study area, so it is kept visually separate; the overall panel shows it both **out-of-fold** (the honest estimate) and **in-sample** (trained and tested on the same labels).

The stratified analysis uses this study's **out-of-fold** prediction, because the mapping model trained on every labelled parcel and its
in-sample verdict there is optimistic. The cross-study agreement instead spans every **in-domain** parcel and uses this study's **deployed**  verdict, the only one defined off the labelled parcels.

Reads `results/comparison/<latest>/`; run `scripts/run_product_comparison.py` (after the masks and `run_final_inference.py`) to populate it.

In [ ]:
NOTEBOOK = "011_existing_product_comparison"

import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

from utils.comparison import (
    agreement_by_reference,
    consensus_breakdown,
    fleiss_kappa,
    pairwise_agreement,
    per_study_agreement_counts,
)
from utils.paths import find_repo_root, get_project_paths
from utils.style import get_figure_size, save_figure, use_publication_style
from utils.terminology import COMPARISON_STUDIES, PALETTE_CATEGORICAL

# Only ``utils*`` is pip-installed, so put the repo root on the path to import ``scripts.*`` below.
if str(find_repo_root()) not in sys.path:
    sys.path.insert(0, str(find_repo_root()))

use_publication_style()
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
    }
)


def _authors_only(label):
    """Keep the author citation only; drop the organisation/product tag (Greenpeace, PRIMOFARO)."""
    return label.replace(", Greenpeace", "").replace(", PRIMOFARO", "")


ORDER = [*COMPARISON_STUDIES, "ratsakatika"]
DISPLAY = {
    **{k: _authors_only(v.display) for k, v in COMPARISON_STUDIES.items()},
    "ratsakatika": "This study",
}
# Two-line labels for tight legends and axes (only the long citations wrap).
DISPLAY_2L = {
    "sabatini": "Sabatini (2020)",
    "munteanu": "Munteanu et al. (2022)",
    "kathmann": "Kathmann et al.\n(2017)",
    "schickhofer": "Schickhofer and\nSchwarz (2019)",
    "ratsakatika": "This study",
}
# Compact one-line labels (author + year) for scatter-point annotations and tight legends.
DISPLAY_SHORT = {
    "sabatini": "Sabatini (2020)",
    "munteanu": "Munteanu et al. (2022)",
    "kathmann": "Kathmann et al. (2017)",
    "schickhofer": "Schickhofer and Schwarz (2019)",
    "ratsakatika": "This study",
}
# Distinct colour per product; this study in blue for emphasis.
STUDY_COLOURS = {
    "sabatini": PALETTE_CATEGORICAL["orange"],
    "munteanu": PALETTE_CATEGORICAL["teal"],
    "kathmann": PALETTE_CATEGORICAL["light_green"],
    "schickhofer": PALETTE_CATEGORICAL["magenta"],
    "ratsakatika": PALETTE_CATEGORICAL["blue"],
}

paths = get_project_paths()
root = paths.results / "comparison"
runs = sorted((p for p in root.iterdir() if p.is_dir()), reverse=True) if root.is_dir() else []
run = next((p for p in runs if (p / "overall_performance.csv").is_file()), None)


def _read(name):
    return pd.read_csv(run / name) if run is not None and (run / name).is_file() else None


performance = _read("overall_performance.csv")
stratified = _read("stratified_performance.csv")
gpkg_path = run / "product_parcel_comparison.gpkg" if run is not None else None

if run is None:
    print("No comparison run yet; run scripts/run_product_comparison.py.")
else:
    print(f"[load] comparison run {run.name}")

## Overall performance against the reference

This study (blue) is shown out-of-fold; its in-sample row is in the table but not plotted as a like-for-like bar.

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask

from scripts.build_existing_product_masks import native_ogf_area_ha
from utils.inference import PIXEL_AREA_HA

# Study metadata for the presentation table: geographic extent of the product, method family, and
# the forest type it targets (each product's own definition; this study targets old-growth).
PRODUCT_META = {
    "sabatini": ("Europe", "Model-based", "Primary"),
    "munteanu": ("Romania", "Model-based", "High conservation value"),
    "kathmann": ("Romania", "Rule-based", "Primary & old-growth"),
    "schickhofer": ("Romania", "Rule-based", "Primary & old-growth"),
    "ratsakatika": ("Făgăraș", "Model-based", "Old-growth"),
}

if performance is not None:
    show = performance.copy()
    show["product"] = show["product"].map(DISPLAY)
    mine = show[show["kind"] != "existing"]

    # Two old-growth-area readings per product, both clipped to the AOI. The main table shows only
    # the parcel reading (consistent with the parcel-level metrics beside it); the native reading is
    # kept for the secondary parcel-vs-native table below (for the results discussion).
    #  * parcel: the summed geometric area of the whole parcels the product calls old-growth,
    #    from the comparison run's parcel table (ogf_<study>). The published GeoPackage
    #    carries only this study's own columns, so the per-product decisions are taken
    #    from the internal table they are built from -- the same source, and the same
    #    parcels, as the footprint column of the area table below.
    #  * native: each product's own binary decision in its native form -- native-resolution raster
    #    pixels for Sabatini/Munteanu, dissolved vector polygons for Kathmann/Schickhofer, and this
    #    study's published 10 m binary raster.
    final_dir = paths.results / "final"
    binary_tif = final_dir / "ogf_binary_3035_10m.tif"
    aoi_gdf = gpd.read_file(paths.aoi)
    aoi_ha = float(aoi_gdf.to_crs(3035).area.sum() / 1e4)
    raw_root = paths.repo_root / "data/raw/existing_products"

    def _native_ogf_ha(study):
        if study != "ratsakatika":  # native raster pixels / dissolved vector polygons in the AOI
            return native_ogf_area_ha(study, aoi_gdf, raw_root)
        with rasterio.open(
            binary_tif
        ) as src:  # this study: published 10 m binary raster in the AOI
            px = rio_mask(
                src, aoi_gdf.to_crs(src.crs).geometry.values, crop=True, filled=True, nodata=0
            )[0][0]
        return float((px == 1).sum()) * PIXEL_AREA_HA

    g_parcels = gpd.read_file(gpkg_path)
    g_parcels["area_ha"] = g_parcels.to_crs(3035).geometry.area / 1e4

    def _parcel_ogf_ha(study):
        verdict = g_parcels[f"ogf_{study}"].astype(bool)
        return float(g_parcels.loc[verdict, "area_ha"].sum())

    def _metric(study, name):
        kind = "out_of_fold" if study == "ratsakatika" else "existing"
        row = performance[(performance["product"] == study) & (performance["kind"] == kind)]
        return float(row[name].iloc[0]) if not row.empty else np.nan

    # OGF prevalence in each product's evaluation set -- the PR-AUC no-skill baseline. Every product
    # is scored on the same common set, so these should be identical; they are computed per row
    # (existing products from the common set, this study from its out-of-fold parcels) rather than
    # assumed, so any divergence in denominators would surface in the table.
    eval_path = run / "product_eval_parcels.parquet"
    common_prevalence = (
        float(pd.read_parquet(eval_path)["reference_label"].astype(bool).mean())
        if eval_path.is_file()
        else np.nan
    )
    _oof_runs = sorted((paths.results / "main_nested_cv").glob("*__xgboost__baseline_tessera"))
    oof_prevalence = np.nan
    if _oof_runs:
        _oof = pd.concat(
            [
                pd.read_parquet(p)
                for p in sorted(_oof_runs[-1].glob("parcel_predictions_fold*.parquet"))
            ]
        )
        oof_prevalence = float(_oof["y_true"].mean())

    def _prevalence(study):
        return oof_prevalence if study == "ratsakatika" else common_prevalence

    rows, area_rows = [], []
    for s in ORDER:
        native, parcel = _native_ogf_ha(s), _parcel_ogf_ha(s)
        extent, method, target = PRODUCT_META[s]
        rows.append(
            {
                "Product": DISPLAY[s],
                "Extent": extent,
                "Method": method,
                "Forest target": target,
                "Parcel OGF area (ha)": f"{parcel:,.0f}",
                "Parcel OGF area (% AOI)": f"{100 * parcel / aoi_ha:.1f}",
                "PR-AUC": f"{_metric(s, 'pr_auc'):.2f}",
                "ROC-AUC": f"{_metric(s, 'roc_auc'):.2f}",
                "F1": f"{_metric(s, 'f1'):.2f}",
                "Precision": f"{_metric(s, 'precision'):.2f}",
                "Recall": f"{_metric(s, 'recall'):.2f}",
                "OGF prevalence": f"{_prevalence(s):.2f}",
            }
        )
        area_rows.append(
            {
                "Product": DISPLAY[s],
                "Parcel (ha)": f"{parcel:,.0f}",
                "Parcel (% AOI)": f"{100 * parcel / aoi_ha:.1f}",
                "Native (ha)": f"{native:,.0f}",
                "Native (% AOI)": f"{100 * native / aoi_ha:.1f}",
                "Native - Parcel (ha)": f"{native - parcel:+,.0f}",
                "Native - Parcel (%)": f"{100 * (native - parcel) / parcel:+.1f}",
            }
        )
    products_table = pd.DataFrame(rows)
    area_table = pd.DataFrame(area_rows)
    n_lab = int(performance[performance["kind"] == "existing"]["n"].iloc[0])
    print(
        "[Existing products] their published predictions, untuned; this study out-of-fold "
        f"(n = {n_lab:,} labelled parcels; AOI = {aoi_ha:,.0f} ha):"
    )
    print(products_table.to_string(index=False))

    # Secondary table (NOT for the manuscript table): parcel-verdict area vs each product's native
    # binary decision, absolute and % of AOI, with the difference -- for the results-section
    # discussion of why the two area conventions diverge (chiefly for this study).
    print(
        "\n[Parcel vs native (pixel) OGF area] not in the main table -- for the discussion. "
        "Parcel = whole parcels the product calls OGF; Native = each product's own binary decision "
        "(this study = 10 m pixel raster); both AOI-clipped."
    )
    print(area_table.to_string(index=False))

    print("\n[This study] (tuned on the AOI; out-of-fold is the honest estimate):")
    print(
        mine[
            [
                "kind",
                "n",
                "pr_auc",
                "roc_auc",
                "f1",
                "precision",
                "recall",
                "f1_inner",
                "precision_inner",
                "recall_inner",
            ]
        ]
        .round(3)
        .to_string(index=False)
    )

    # Histogram-overlap coefficient per product: the overlap of the true-old-growth and
    # true-non-old-growth score distributions (this study's out-of-fold probability, each existing
    # product's parcel old-growth fraction) on the labelled parcels. 0 = perfectly separated,
    # 1 = identical, so lower is better. Same scores and bins as the predicted-by-reference figure.
    def _overlap_coefficient(a, b, bins):
        da, _ = np.histogram(np.asarray(a, dtype=float), bins=bins, density=True)
        db, _ = np.histogram(np.asarray(b, dtype=float), bins=bins, density=True)
        return float(np.minimum(da, db).sum() * (bins[1] - bins[0]))

    overlap = dict.fromkeys(ORDER, np.nan)
    if gpkg_path is not None and gpkg_path.is_file():
        g_lab = gpd.read_file(gpkg_path)
        g_lab = g_lab[g_lab["reference_label"].notna()].copy()
        oof_runs = sorted((paths.results / "main_nested_cv").glob("*__xgboost__baseline_tessera"))
        if oof_runs:
            oof = pd.concat(
                [
                    pd.read_parquet(p)
                    for p in sorted(oof_runs[-1].glob("parcel_predictions_fold*.parquet"))
                ]
            )
            g_lab["ratsakatika_oof"] = g_lab["parcel_id"].map(oof.set_index("parcel_id")["p_mean"])
        else:
            g_lab["ratsakatika_oof"] = g_lab["ogf_ratsakatika_probability"]
        ref_b = g_lab["reference_label"].astype(bool)
        bins = np.linspace(0, 1, 21)
        score_col = {
            "ratsakatika": "ratsakatika_oof",
            **{s: f"ogf_{s}_frac" for s in COMPARISON_STUDIES},
        }
        for s in ORDER:
            pos = g_lab.loc[ref_b, score_col[s]].dropna()
            neg = g_lab.loc[~ref_b, score_col[s]].dropna()
            overlap[s] = _overlap_coefficient(pos, neg, bins)

    show = performance[performance["kind"].isin(["existing", "out_of_fold"])].copy()
    show["overlap"] = show["product"].map(overlap)
    panels = [
        ("pr_auc", "a) PR-AUC"),
        ("roc_auc", "b) ROC-AUC"),
        ("f1", "c) F1"),
        ("precision", "d) Precision"),
        ("recall", "e) Recall"),
        ("overlap", "f) Overlap (lower better)"),
    ]
    fig, axes = plt.subplots(
        2, 3, figsize=get_figure_size("double", aspect=0.58), constrained_layout=True
    )
    for ax, (metric, title) in zip(axes.ravel(), panels, strict=False):
        rows, labels, colours = [], [], []
        for study in ORDER:
            sub = show[show["product"] == study]
            if sub.empty:
                continue
            rows.append(float(sub[metric].iloc[0]))
            labels.append(DISPLAY_2L[study])
            colours.append(PALETTE_CATEGORICAL["blue"] if study == "ratsakatika" else "0.6")
        ypos = np.arange(len(rows))
        ax.barh(ypos, rows, color=colours)
        for yi, value in zip(ypos, rows, strict=False):
            ax.text(value + 0.02, yi, f"{value:.2f}", va="center", fontsize=6)
        ax.axvline(0, color="0.4", lw=0.6)
        ax.set_yticks(ypos)
        ax.set_yticklabels(labels if metric in ("pr_auc", "precision") else [], fontsize=6.5)
        ax.set_xlim(0.0, 1.12)
        ax.tick_params(labelsize=7)
        ax.set_title(title, loc="left", fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Performance vs reference (this study in blue, out-of-fold)", fontsize=10.5)
    save_figure(fig, f"{NOTEBOOK}/overall_performance", data=performance)
else:
    print("overall_performance.csv not found.")

## Relative differences between the existing products

In [ ]:
import geopandas as gpd
from sklearn import metrics as skm

from utils.bootstrap import block_bootstrap_distribution, percentile_interval
from utils.terminology import BOOTSTRAP_REPS_INFERENCE, SEED

# PR-AUC is the average precision, as in the manuscript; F1/precision/recall use the 0.5
# area-fraction majority rule, the same operating point as the table above.
PRODUCT_METRICS = {
    "PR-AUC": lambda y, s: skm.average_precision_score(y, s),
    "ROC-AUC": lambda y, s: skm.roc_auc_score(y, s),
    "F1": lambda y, s: skm.f1_score(y, s >= 0.5, zero_division=0.0),
    "Precision": lambda y, s: skm.precision_score(y, s >= 0.5, zero_division=0.0),
    "Recall": lambda y, s: skm.recall_score(y, s >= 0.5, zero_division=0.0),
}

_diff_path = run / "product_eval_parcels.parquet" if run is not None else None
if _diff_path is None or not _diff_path.is_file():
    print("Run scripts/run_product_comparison.py for the common-set evaluation table.")
else:
    _ev = pd.read_parquet(_diff_path)
    _lab = gpd.read_file(
        paths.repo_root / "data/processed/vectors/labels/ogf_reference_labels_partitioned.gpkg"
    )
    _lab["parcel_id"] = _lab["parcel_id"].astype("int64")
    _ev = _ev.merge(_lab[["parcel_id", "bootstrap_id"]], on="parcel_id", how="left")

    _keys = list(COMPARISON_STUDIES)
    _y = _ev["reference_label"].astype(int).to_numpy()
    _blocks = _ev["bootstrap_id"].to_numpy()
    _scores = np.column_stack([_ev[f"ogf_{s}_frac"].fillna(0.0).to_numpy() for s in _keys])

    point_estimates = pd.DataFrame(
        {
            metric: {s: fn(_y, _scores[:, j]) for j, s in enumerate(_keys)}
            for metric, fn in PRODUCT_METRICS.items()
        }
    ).loc[_keys]
    point_estimates.insert(0, "Method", [PRODUCT_META[s][1] for s in _keys])
    point_estimates.index = [DISPLAY[s] for s in _keys]
    print(f"[Point estimates] common set n = {len(_ev):,}, OGF prevalence = {_y.mean():.3f}:")
    print(point_estimates.round(3).to_string())

    # Pairwise relative difference on the primary metric.
    _pr = {s: PRODUCT_METRICS["PR-AUC"](_y, _scores[:, j]) for j, s in enumerate(_keys)}
    pairwise_pct = pd.DataFrame(
        [[(_pr[r] / _pr[c] - 1) * 100 if r != c else np.nan for c in _keys] for r in _keys],
        index=[DISPLAY[s] for s in _keys],
        columns=[DISPLAY[s].split(" ")[0] for s in _keys],
    )
    print("\n[Pairwise % difference in PR-AUC]  (row - column) / column * 100:")
    print(pairwise_pct.round(1).to_string())

    # Best of each method family, contrasted on identical block resamples so the difference is
    # paired. The bootstrap mean is reported beside the point estimate to show they agree.
    _rule = max((s for s in _keys if PRODUCT_META[s][1] == "Rule-based"), key=lambda s: _pr[s])
    _model = max((s for s in _keys if PRODUCT_META[s][1] == "Model-based"), key=lambda s: _pr[s])
    _ir, _im = _keys.index(_rule), _keys.index(_model)
    _rows = []
    for _name, _fn in PRODUCT_METRICS.items():
        _dist = block_bootstrap_distribution(
            _y,
            np.column_stack([_scores[:, _ir], _scores[:, _im]]),
            _blocks,
            _fn,
            n_reps=BOOTSTRAP_REPS_INFERENCE,
            seed=SEED,
        )
        _a, _b = _dist[:, 0], _dist[:, 1]
        _pa, _pb = _fn(_y, _scores[:, _ir]), _fn(_y, _scores[:, _im])
        _lo, _hi = percentile_interval((_a / _b - 1) * 100)
        _alo, _ahi = percentile_interval(_a - _b)
        _rows.append(
            {
                "Metric": _name,
                "Rule-based": round(_pa, 3),
                "Model-based": round(_pb, 3),
                "% diff (point)": round((_pa / _pb - 1) * 100, 1),
                "% diff (boot mean)": round((_a.mean() / _b.mean() - 1) * 100, 1),
                "% 95% CI": f"[{_lo:+.1f}, {_hi:+.1f}]",
                "CI excludes 0": "yes" if (_alo > 0 or _ahi < 0) else "no",
            }
        )
    family_diff = pd.DataFrame(_rows)
    print(f"\n[Best rule-based vs best model-based] {DISPLAY[_rule]} vs {DISPLAY[_model]}:")
    print(family_diff.to_string(index=False))

    # If the point and bootstrap-mean magnitudes agree, the point-estimate table is enough for the
    # numbers quoted in the text and the bootstrap is only needed for the intervals.
    _gap = (family_diff["% diff (boot mean)"] - family_diff["% diff (point)"]).abs().max()
    print(
        f"\nLargest gap between the point and bootstrap-mean % difference: {_gap:.2f} pp. "
        "The two agree, so the point estimates carry the magnitudes; the bootstrap is needed "
        "only for the intervals."
    )

    # The method families interleave, so 'rule-based beats model-based' is NOT supported as a
    # family-level claim -- only the best-of-each contrast above is.
    print("\n[Family check] products ranked by PR-AUC:")
    for _s in sorted(_keys, key=lambda s: -_pr[s]):
        print(f"  {DISPLAY[_s]:32s} {PRODUCT_META[_s][1]:12s} PR-AUC = {_pr[_s]:.3f}")

## Stratified performance

In [ ]:
import geopandas as gpd
from sklearn.metrics import roc_auc_score

from utils.bootstrap import block_bootstrap_distribution, percentile_interval
from utils.terminology import BOOTSTRAP_REPS_INFERENCE, FEATURE_SET_BANDS, NODATA, SEED

# --- Stratified-bar styling (tweak these) ---------------------------------------
EXISTING_ALPHA = 0.6  # opacity of the four existing-product bars (1.0 = opaque)
GROUP_GAP = 0.5  # gap between the existing bars and this study's bar, in bar widths
# --------------------------------------------------------------------------------

_eval_path = run / "product_eval_parcels.parquet" if run is not None else None
if _eval_path is None or not _eval_path.is_file():
    print("Run scripts/run_product_comparison.py for the common-set evaluation table.")
else:
    # One common evaluation set for every product (the parcels this study scored out-of-fold), with
    # the spatial block id used by the 20-block bootstrap and the CORINE forest type joined on.
    ev = pd.read_parquet(_eval_path)
    _labels = gpd.read_file(
        paths.repo_root / "data/processed/vectors/labels/ogf_reference_labels_partitioned.gpkg"
    )
    _labels["parcel_id"] = _labels["parcel_id"].astype("int64")
    ev = ev.merge(
        _labels[["parcel_id", "bootstrap_id", "corine_forest_type"]], on="parcel_id", how="left"
    )

    # Parcel-mean terrain covariates, averaged over the labelled-pixel cache (which already covers
    # exactly this common parcel set). The NoData sentinel is masked before averaging, as in
    # scripts/run_product_comparison.parcel_mean_elevation -- otherwise it corrupts the means.
    _idx = pd.read_parquet(paths.results / "main_nested_cv" / "pixel_index.parquet")
    _feats = np.load(paths.cache / "pixel_features" / "baseline_tessera.npy", mmap_mode="r")
    _bands = list(FEATURE_SET_BANDS["baseline_tessera"])
    COVARIATES = {
        "elevation_m": "(b) Altitude tertile",
        "slope_deg": "(c) Slope tertile",
    }
    _cov = pd.DataFrame({"parcel_id": _idx["parcel_id"].to_numpy()})
    for _name in COVARIATES:
        _values = np.asarray(_feats[:, _bands.index(_name)], dtype=np.float64)
        _values[_values == NODATA] = np.nan  # NoData must not enter the parcel mean
        _cov[_name] = _values
    ev = ev.merge(_cov.groupby("parcel_id").mean(), on="parcel_id", how="left")

    def _tertiles(series):
        """Equal-count tertiles, labelled by their value range (units live in the panel title)."""
        q = series.quantile([0, 1 / 3, 2 / 3, 1]).to_numpy()
        # En dash between the bounds and a thousands separator, so the altitude ticks read
        # "535\u20131,109 m"; the slope ticks take the same dash for consistency.
        labels = [f"{q[i]:,.0f}\u2013{q[i + 1]:,.0f}" for i in range(3)]
        binned = pd.cut(
            series, bins=[-np.inf, q[1], q[2], np.inf], labels=labels, include_lowest=True
        )
        return binned.astype("object"), labels

    ev["forest_type"] = ev["corine_forest_type"]
    strata = [("(a) Forest type", "forest_type", ["broadleaf", "coniferous", "mixed"])]
    for _name, _title in COVARIATES.items():
        _col = f"{_name}_tertile"
        ev[_col], _labs = _tertiles(ev[_name])
        strata.append((_title, _col, _labs))

    # Display-only tick text: forest types capitalised, tertile ranges carry their unit.
    STRATUM_VALUE_DISPLAY = {
        "broadleaf": "Broadleaf",
        "coniferous": "Coniferous",
        "mixed": "Mixed",
    }
    # Altitude takes a spaced unit ("1,109 m"); the degree sign closes up to the number.
    STRATUM_UNIT = {"(b) Altitude tertile": " m", "(c) Slope tertile": "\u00b0"}

    SCORE_COL = {
        "ratsakatika": "ratsakatika_oof",
        **{s: f"ogf_{s}_frac" for s in COMPARISON_STUDIES},
    }
    _ref = ev["reference_label"].astype(int).to_numpy()

    # ROC-AUC per stratum x product, with a 95% interval from the same 20-block spatial bootstrap
    # used for the headline results in notebook 009 (blocks resampled with replacement; all five
    # products evaluated on identical resamples, so the bars are comparable within a panel).
    strat_rows = []
    for _title, _col, _values in strata:
        for _v in _values:
            _m = (ev[_col] == _v).to_numpy()
            _y = _ref[_m]
            if _y.min() == _y.max():  # ROC-AUC undefined without both classes
                continue
            _scores = np.column_stack(
                [ev.loc[_m, SCORE_COL[k]].fillna(0.0).to_numpy() for k in ORDER]
            )
            _dist = block_bootstrap_distribution(
                _y,
                _scores,
                ev.loc[_m, "bootstrap_id"].to_numpy(),
                lambda a, b: roc_auc_score(a, b),
                n_reps=BOOTSTRAP_REPS_INFERENCE,
                seed=SEED,
            )
            for _j, _k in enumerate(ORDER):
                _lo, _hi = percentile_interval(_dist[:, _j])
                strat_rows.append(
                    {
                        "stratum": _title,
                        "value": _v,
                        "product": _k,
                        "n": int(_m.sum()),
                        "prevalence": float(_y.mean()),
                        "roc_auc": float(roc_auc_score(_y, _scores[:, _j])),
                        "ci_lo": _lo,
                        "ci_hi": _hi,
                    }
                )
    strat_roc = pd.DataFrame(strat_rows)
    print("[Stratified ROC-AUC] with 95% 20-block spatial bootstrap intervals:")
    _show = strat_roc.copy()
    _show["product"] = _show["product"].map(DISPLAY)
    print(_show.round(3).to_string(index=False))
    # The forest-type panel covers fewer parcels than the tertile panels: 65 of the 4,825 common
    # parcels have no CORINE forest type, whereas every parcel has a continuous covariate value.
    print("\nParcels per panel (they do NOT all match -- see the CORINE coverage gap):")
    print(strat_roc.drop_duplicates(["stratum", "value"]).groupby("stratum")["n"].sum().to_string())

    fig, axes = plt.subplots(
        1,
        len(strata),
        # 15% shorter than the aspect-0.33 original; the cut comes out of the axes
        figsize=get_figure_size("double", aspect=0.33 * 0.85),
        sharey=True,
        constrained_layout=True,
    )
    width = 0.16
    for ax, (title, _col, values) in zip(axes, strata, strict=True):
        present = [
            v
            for v in values
            if not strat_roc[(strat_roc["stratum"] == title) & (strat_roc["value"] == v)].empty
        ]
        x = np.arange(len(present))
        for offset, study in enumerate(ORDER):
            sub = (
                strat_roc[(strat_roc["stratum"] == title) & (strat_roc["product"] == study)]
                .set_index("value")
                .reindex(present)
            )
            vals = sub["roc_auc"].to_numpy(dtype=float)
            # Nudge the existing bars left and this study's bar right by half the gap each,
            # so the five-bar cluster stays centred on the tick while this study sits apart.
            _shift = (GROUP_GAP / 2) * width
            if study == "ratsakatika":
                # This study keeps its 95% bootstrap interval and gets a black outline.
                err = np.vstack(
                    [vals - sub["ci_lo"].to_numpy(float), sub["ci_hi"].to_numpy(float) - vals]
                )
                ax.bar(
                    x + (offset - 2) * width + _shift,
                    vals,
                    width,
                    color=STUDY_COLOURS[study],
                    edgecolor="black",
                    linewidth=0.6,
                    yerr=err,
                    error_kw={"ecolor": "black", "elinewidth": 0.7, "capsize": 1.5},
                )
            else:
                # Existing products: no CI bars, slightly transparent to recede visually.
                ax.bar(
                    x + (offset - 2) * width - _shift,
                    vals,
                    width,
                    color=STUDY_COLOURS[study],
                    alpha=EXISTING_ALPHA,
                )
        meta = (
            strat_roc[strat_roc["stratum"] == title]
            .drop_duplicates("value")
            .set_index("value")
            .reindex(present)
        )
        ax.set_xticks(x)
        ax.set_xticklabels(
            [
                f"{STRATUM_VALUE_DISPLAY.get(v, v)}{STRATUM_UNIT.get(title, '')}\n"
                f"n = {int(meta.loc[v, 'n']):,}\n"
                f"OGF prev = {meta.loc[v, 'prevalence']:.2f}"
                for v in present
            ],
            # 5.9, not 6.2: the spaced "OGF prev = 0.29" line leaves under a point of white
            # between neighbouring ticks at the larger size.
            fontsize=5.9,
            linespacing=1.15,
        )
        ax.set_ylim(0.5, 1.0)  # 0.5 is the ROC-AUC no-skill floor; nothing is clipped by it
        ax.tick_params(axis="x", length=0, pad=1.5)
        ax.set_title(title, loc="left", fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)

    # Tick values only on the leftmost axis; the floor is labelled as the no-skill point.
    axes[0].set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
    axes[0].set_yticklabels(["0.5 (no skill)", "0.6", "0.7", "0.8", "0.9", "1.0"], fontsize=7)
    axes[0].set_ylabel("ROC-AUC", fontsize=8)
    for ax in axes[1:]:
        ax.tick_params(axis="y", labelleft=False, length=0)

    strat_labels = {**DISPLAY_SHORT, "ratsakatika": "This study (out-of-fold)"}
    handles = [
        Line2D(
            [0],
            [0],
            marker="s",
            color=STUDY_COLOURS[s],
            lw=0,
            label=strat_labels[s],
            alpha=1.0 if s == "ratsakatika" else EXISTING_ALPHA,
            markeredgecolor="black" if s == "ratsakatika" else "none",
            markeredgewidth=0.6 if s == "ratsakatika" else 0,
        )
        for s in ORDER
    ]
    fig.legend(
        handles=handles,
        loc="outside lower center",
        ncol=len(ORDER),
        frameon=False,
        fontsize=6.5,
        borderaxespad=0.0,
        handletextpad=0.4,
        columnspacing=1.2,
    )
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.0, wspace=0.03, hspace=0.0)
    save_figure(fig, f"{NOTEBOOK}/fig_3_stratified", data=strat_roc)

## Cross-study agreement

In [ ]:
import geopandas as gpd

comparison = gpd.read_file(gpkg_path) if gpkg_path is not None and gpkg_path.is_file() else None
if comparison is not None:
    # Agreement is over the wall-to-wall predictions, so use every in-domain parcel (not just the
    # labelled ones); this study's deployed verdict applies everywhere it has a prediction.
    evaluable = comparison[comparison["ogf_ratsakatika_probability"].notna()].copy()
    verdict_cols = [f"ogf_{s}" for s in ORDER]
    labelled = evaluable[evaluable["reference_label"].notna()].copy()
    labelled["reference_label"] = labelled["reference_label"].astype(bool)
    print(
        f"[Consensus] {len(evaluable):,} in-domain parcels: all agree old-growth / all non / mixed:"
    )
    print(
        pd.Series(
            {
                **consensus_breakdown(evaluable, verdict_cols),
                "fleiss_kappa": round(fleiss_kappa(evaluable, verdict_cols), 3),
            }
        ).to_string()
    )
    by_ref = agreement_by_reference(labelled, verdict_cols, "reference_label")
    by_ref["product"] = [DISPLAY.get(p.replace("ogf_", ""), p) for p in by_ref["product"]]
    print("\n[Agreement by reference] fraction calling old-growth, by reference class (labelled):")
    print(by_ref.round(3).to_string(index=False))
else:
    print("No comparison GeoPackage; run scripts/run_product_comparison.py.")

In [ ]:
if comparison is not None:
    evaluable = comparison[comparison["ogf_ratsakatika_probability"].notna()].copy()
    verdict_cols = [f"ogf_{s}" for s in ORDER]
    names = [DISPLAY_2L[s] for s in ORDER]

    # Top row: every in-domain parcel. Bottom row: the unlabelled parcels only (neither an
    # OGF nor a non-OGF reference label), the stratum where no product is validated and
    # cross-study agreement is the only consistency check available.
    strata = [
        ("all", "All in-domain parcels", evaluable),
        ("unlabelled", "Unlabelled parcels only", evaluable[evaluable["reference_label"].isna()]),
    ]
    pairwise = {k: pairwise_agreement(f, verdict_cols) for k, _, f in strata}
    per_study = {k: per_study_agreement_counts(f, verdict_cols) for k, _, f in strata}

    # One shared colour scale, so the two matrices can be read against each other.
    off_diagonal = ~np.eye(len(verdict_cols), dtype=bool)
    vmin = min(float(m.to_numpy()[off_diagonal].min()) for m in pairwise.values())

    fig = plt.figure(figsize=get_figure_size("double", aspect=1.0), constrained_layout=True)
    # One column per stratum, so the two are read against each other across a row: the
    # pairwise-agreement matrices side by side on the top row, the mutual-agreement curves
    # side by side beneath them. The narrow right-hand column carries the colourbar shared
    # by both matrices, and set_box_aspect keeps all four panels equal-sized squares.
    gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.0, 0.05])
    panels = iter("abcd")
    image = None
    chart_axes = []

    # Top row: pairwise-agreement matrix per stratum, with the fractions printed in each cell.
    for col, (key, stratum, frame) in enumerate(strata):
        ax_m = fig.add_subplot(gs[0, col])
        # Two-line titles: the stratum and its parcel count do not fit on one line at this
        # width, and giving all four panels two lines keeps the titles aligned across a row.
        subtitle = f"{stratum} (n = {len(frame):,})"
        mat = pairwise[key].reindex(index=verdict_cols, columns=verdict_cols).to_numpy()
        image = ax_m.imshow(mat, cmap="YlGnBu", vmin=vmin, vmax=1.0)
        ax_m.set_xticks(range(len(names)))
        ax_m.set_xticklabels(names, rotation=30, ha="right", fontsize=6.5)
        ax_m.set_yticks(range(len(names)))
        ax_m.set_yticklabels(names, fontsize=6.5)
        for i in range(len(names)):
            for j in range(len(names)):
                ax_m.text(
                    j,
                    i,
                    f"{mat[i, j]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="white" if mat[i, j] > 0.85 else "black",
                )
        ax_m.set_title(f"{next(panels)}) Pairwise agreement\n{subtitle}", loc="left", fontsize=9)
        ax_m.set_box_aspect(1)

    # Bottom row: for each product, how many of the other four share its classification.
    for col, (key, stratum, frame) in enumerate(strata):
        ax_c = fig.add_subplot(gs[1, col])
        subtitle = f"{stratum} (n = {len(frame):,})"
        for study in ORDER:
            sub = per_study[key][per_study[key]["product"] == f"ogf_{study}"].sort_values("n_agree")
            ax_c.plot(
                sub["n_agree"],
                sub["n_parcels"],
                "-o",
                color=STUDY_COLOURS[study],
                lw=1.4,
                ms=3.5,
                label=DISPLAY_SHORT[study],
            )
        ax_c.set_xlabel("Other products sharing the classification (of 4)", fontsize=8)
        ax_c.set_ylabel("Parcels", fontsize=8)
        ax_c.set_xticks(range(5))
        ax_c.tick_params(labelsize=7)
        ax_c.set_title(f"{next(panels)}) Mutual agreement\n{subtitle}", loc="left", fontsize=9)
        # Keep the full frame (not the usual open spines) so this panel reads as the same
        # square as the matrix above it.
        for spine in ax_c.spines.values():
            spine.set_linewidth(0.8)
        ax_c.set_box_aspect(1)
        chart_axes.append(ax_c)

    fig.colorbar(image, cax=fig.add_subplot(gs[0, 2]), label="Fraction of parcels agreeing")
    # Handles from a curve panel, not fig.axes[1], which is a matrix in this layout.
    handles, labels = chart_axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=3, frameon=False, fontsize=7.5)
    fig.suptitle("Cross-study agreement (this study's deployed classification)", fontsize=10.5)
    save_figure(
        fig,
        f"{NOTEBOOK}/fig_s9_agreement",
        data={f"pairwise_{k}": pairwise[k].reset_index(names="product") for k, _, _ in strata}
        | {f"per_study_{k}": per_study[k] for k, _, _ in strata},
    )

    # High-confidence parcels of this study: do the other products agree?
    high = evaluable[evaluable["ogf_ratsakatika_probability"] >= 0.8]
    if len(high) > 0:
        others = [f"ogf_{s}" for s in COMPARISON_STUDIES]
        agree_frac = high[others].mean()
        print(f"\n[High-confidence] {len(high)} parcels with calibrated probability >= 0.8:")
        print("Fraction of these the existing products also call old-growth:")
        print(agree_frac.rename(lambda s: DISPLAY[s.replace("ogf_", "")]).round(3).to_string())
else:
    print("Agreement tables or comparison GeoPackage not found.")

## Predicted old-growth from the continuous product surfaces

In [ ]:
def overlap_coefficient(a, b, bins):
    """Overlap coefficient of two samples' densities (0 = disjoint, 1 = identical)."""
    da, _ = np.histogram(np.asarray(a, dtype=float), bins=bins, density=True)
    db, _ = np.histogram(np.asarray(b, dtype=float), bins=bins, density=True)
    return float(np.minimum(da, db).sum() * (bins[1] - bins[0]))


# The common evaluation set: the parcels this study scored out-of-fold, with each product
# re-aggregated over the same eroded pixel geometry (product_eval_parcels.parquet).
eval_path = run / "product_eval_parcels.parquet" if run is not None else None
lab = pd.read_parquet(eval_path) if eval_path is not None and eval_path.is_file() else None
cont_path = run / "continuous_product_parcels.csv" if run is not None else None
if lab is None or cont_path is None or not cont_path.is_file():
    print("Run scripts/build_continuous_product_parcels.py for the continuous-surface figure.")
else:
    merged = lab.merge(pd.read_csv(cont_path), on="parcel_id", how="left")
    mref = merged["reference_label"].astype(bool)
    cn_ogf, cn_non = int(mref.sum()), int((~mref).sum())
    cprevalence = float(mref.mean())
    cpanels = [
        ("This study", "ratsakatika_oof", "OOF probability"),
        ("Sabatini (2020)", "sabatini_probability", "BRT probability"),
        ("Munteanu et al. (2022)", "munteanu_complexity", "Complexity index"),
    ]
    fig, axes = plt.subplots(
        1, 3, figsize=get_figure_size("double", aspect=0.42), sharey=True, constrained_layout=True
    )
    for ax, (title, col, xlab) in zip(axes, cpanels, strict=True):
        # NA = 0 (non-old-growth); restricted to the common set via lab, so n is identical
        # across the three panels.
        pos, neg = merged.loc[mref, col].fillna(0.0), merged.loc[~mref, col].fillna(0.0)
        ax.hist(
            pos,
            bins=bins,
            density=True,
            color=PALETTE_CATEGORICAL["light_green"],
            alpha=0.6,
            label=f"True old-growth (n = {cn_ogf:,})",
        )
        ax.hist(
            neg,
            bins=bins,
            density=True,
            color=PALETTE_CATEGORICAL["orange"],
            alpha=0.45,
            label=f"True non-old-growth (n = {cn_non:,})",
        )
        ax.text(
            0.97,
            0.95,
            f"Overlap coefficient {overlap_coefficient(pos, neg, bins):.2f}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=7,
            color="0.3",
        )
        ax.set_title(title, loc="left", fontsize=9)
        ax.set_xlabel(xlab, fontsize=8)
        ax.set_xlim(0, 1)
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].set_ylabel("Density", fontsize=8)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=2, frameon=False, fontsize=8)
    # The overlap coefficient is binning-dependent, so the title states the
    # histogram the panels (and the coefficient) use.
    fig.suptitle(
        "Continuous product surfaces (sensitivity check; this study out-of-fold)\n"
        f"Common set n = {len(merged):,}; OGF prevalence {cprevalence:.2f}; "
        f"Overlap coefficient, {len(bins) - 1} bins over 0-1",
        fontsize=11,
    )
    save_figure(
        fig,
        f"{NOTEBOOK}/fig_s5_predicted_by_reference_class_continuous",
        data=merged[["reference_label", *[c for _, c, _ in cpanels]]],
    )

## Threshold sensitivity (supplementary)

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

sweep_path = run / "threshold_sensitivity.csv" if run is not None else None
eval_path2 = run / "product_eval_parcels.parquet" if run is not None else None
if sweep_path is not None and sweep_path.is_file():
    sweep = pd.read_csv(sweep_path)  # run_product_comparison.py (eroded geometry, common set)
elif eval_path2 is not None and eval_path2.is_file():
    # Fallback: compute the sweep from the common-set eval table (existing products on their eroded
    # old-growth area fraction, this study on its out-of-fold probability).
    e = pd.read_parquet(eval_path2)
    ref_sw = e["reference_label"].astype(bool).to_numpy()
    score_col = {
        **{s: f"ogf_{s}_frac" for s in COMPARISON_STUDIES},
        "ratsakatika": "ratsakatika_oof",
    }
    rows = []
    for s in ORDER:
        score = e[score_col[s]].to_numpy(dtype=float)
        for t in np.round(np.arange(0.1, 0.95, 0.1), 2):
            v = score >= t  # NaN >= t is False, i.e. non-old-growth
            rows.append(
                {
                    "product": s,
                    "threshold": float(t),
                    "n": len(e),
                    "f1": float(f1_score(ref_sw, v, zero_division=0.0)),
                    "precision": float(precision_score(ref_sw, v, zero_division=0.0)),
                    "recall": float(recall_score(ref_sw, v, zero_division=0.0)),
                }
            )
    sweep = pd.DataFrame(rows)
else:
    sweep = None

if sweep is None:
    print("No comparison run; run scripts/run_product_comparison.py for the threshold sweep.")
else:
    # Supplementary table: F1 by product across the threshold (0.5 = primary majority rule).
    f1_tab = sweep.pivot_table(index="product", columns="threshold", values="f1").reindex(ORDER)
    f1_tab.index = [DISPLAY[s] for s in f1_tab.index]
    print("[Threshold sensitivity] F1 by product across the decision threshold (0.5 = primary):")
    print(f1_tab.round(3).to_string())

    sweep_labels = {**DISPLAY_SHORT, "ratsakatika": "This study (out-of-fold)"}
    fig, ax = plt.subplots(figsize=get_figure_size("one_half", aspect=0.7), constrained_layout=True)
    for s in ORDER:
        sub = sweep[sweep["product"] == s].sort_values("threshold")
        ax.plot(
            sub["threshold"],
            sub["f1"],
            "-o",
            ms=3,
            lw=1.4,
            color=STUDY_COLOURS[s],
            label=sweep_labels[s],
        )
    ax.axvline(0.5, color="0.6", lw=0.8, ls="--", label="0.5 majority rule")
    ax.set_xlabel("Decision threshold", fontsize=8)
    ax.set_ylabel("F1", fontsize=8)
    ax.set_ylim(0.0, 1.0)
    ax.tick_params(labelsize=7)
    ax.set_title(
        "Threshold sensitivity of F1 (0.5 majority rule dashed; this study out-of-fold)",
        loc="left",
        fontsize=8,
    )
    ax.legend(frameon=False, fontsize=6.5)
    ax.spines[["top", "right"]].set_visible(False)
    save_figure(fig, f"{NOTEBOOK}/fig_s2_threshold_sensitivity", data=sweep)